# Model Selection & Training

This notebook trains multiple classification models, evaluates them, selects the best-performing model, and saves it as a PKL file.

## 1. Import Libraries

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import joblib


## 2. Load Engineered Dataset

In [ ]:

df = pd.read_csv("data/engineered_data.csv")
df.head()


## 3. Train-Test Split

In [ ]:

X = df.drop(columns=["Mood_Score"])
y = df["Mood_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 4. Define Models

In [ ]:

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}


## 5. Model Training & Evaluation

In [ ]:

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy").mean()
    
    results.append({
        "Model": name,
        "Test Accuracy": acc,
        "CV Accuracy": cv_score
    })
    
    print(f"\n{name}")
    print("Accuracy:", acc)
    print(classification_report(y_test, preds))


## 6. Model Comparison

In [ ]:

results_df = pd.DataFrame(results).sort_values(by="CV Accuracy", ascending=False)
results_df


## 7. Select Best Model

In [ ]:

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Best Model Selected:", best_model_name)


## 8. Save Best Model as PKL

In [ ]:

joblib.dump(best_model, "best_model.pkl")
print("Best model saved as best_model.pkl")


## 9. Confusion Matrix of Best Model

In [ ]:

best_preds = best_model.predict(X_test)
confusion_matrix(y_test, best_preds)
